In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from collections import Counter
import ast
import re
import matplotlib.gridspec as gridspec


from matplotlib.colors import LinearSegmentedColormap
from matplotlib import colors as mcolors
from scipy.stats import linregress
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans
from tabulate import tabulate
from collections import Counter

%matplotlib inline


In [ ]:
data_pendapatan=pd.read_csv('../data/data_pendapatan_cleared_mei_november_2023.csv').drop(columns=['Unnamed: 0'])
data_additional=pd.read_excel('../data/customer_data_all_process_v3.xlsx')

In [ ]:
add_data_col_used=data_additional[['Cust No.','Average_Days_Between_Purchases',
       'Day_Of_Week', 'Week_of_Month']].copy()

In [ ]:
data_pendapatan.drop(columns=['Kontribusi in IDR','Kontribusi in USD','Pendapatan in USD'],inplace=True)

In [ ]:
data_pendapatan['Tanggal Nota'].unique()

In [ ]:
data_pendapatan['Tanggal Nota'] = pd.to_datetime(data_pendapatan['Tanggal Nota'])
data_pendapatan_used=data_pendapatan.loc[data_pendapatan['Pendapatan in IDR']>0]
days_between_purchases = data_pendapatan_used.sort_values(by=['Cust No.', 'Tanggal Nota']).groupby('Cust No.')['Tanggal Nota'].diff().dt.days
result = data_pendapatan_used.copy()
result['Days Between Purchases'] = days_between_purchases
result['Days Between Purchases'].fillna(0,inplace=True)

In [ ]:
aggregate_average_days = result.groupby(['Cust No.']).agg(
    average_days_between_transactions=('Days Between Purchases', 'mean'),
).reset_index()
aggregate_average_days

In [ ]:
data_pendapatan_used['Day_Of_Week'] = data_pendapatan_used['Tanggal Nota'].dt.dayofweek
data_pendapatan_used['Week_of_Month'] = data_pendapatan_used['Tanggal Nota'].dt.day // 7 + 1


In [ ]:
favorite_shopping_day = data_pendapatan_used.groupby(['Cust No.', 'Day_Of_Week']).size().reset_index(name='Count')
favorite_shopping_day = favorite_shopping_day.loc[favorite_shopping_day.groupby('Cust No.')['Count'].idxmax()][['Cust No.', 'Day_Of_Week']]

In [ ]:
favorite_shopping_week = data_pendapatan_used.groupby(['Cust No.', 'Week_of_Month']).size().reset_index(name='Count')
favorite_shopping_week = favorite_shopping_week.loc[favorite_shopping_week.groupby('Cust No.')['Count'].idxmax()][['Cust No.', 'Week_of_Month']]

In [ ]:
customer_data = pd.merge(aggregate_average_days, favorite_shopping_day, on='Cust No.')
customer_data = pd.merge(customer_data, favorite_shopping_week, on='Cust No.')
tmp=customer_data.copy()

In [ ]:
tmp

In [ ]:
customer_data = data_pendapatan_used.groupby('Cust No.')['Tanggal Nota'].max().reset_index()
customer_data

In [ ]:
most_recent_date = data_pendapatan_used['Tanggal Nota'].max()
most_recent_date

In [ ]:
customer_data['Tanggal Nota'] = pd.to_datetime(customer_data['Tanggal Nota'])

In [ ]:
customer_data['Recency'] = (most_recent_date - customer_data['Tanggal Nota']).dt.days
customer_data

In [ ]:
customer_data.drop(columns=['Tanggal Nota'],inplace=True)

In [ ]:
customer_data=customer_data.merge(tmp,on='Cust No.', how='left')
customer_data

# RFM Only

In [ ]:
data_pendapatan_used=data_pendapatan.loc[data_pendapatan['Pendapatan in IDR']>0]

In [ ]:
customer_data

In [ ]:
fm_feature = data_pendapatan_used.groupby(['Cust No.']).agg(
    freq_transaction=('Pendapatan in IDR', 'count'),
    total_transaction=('Pendapatan in IDR','sum'),
).reset_index()
fm_feature

In [ ]:
fm_check=fm_feature.loc[fm_feature['freq_transaction']==1]
fm_check

In [ ]:
customer_data = customer_data.merge(fm_feature, on='Cust No.', how='left')

In [ ]:
customer_data.isna().sum()

In [ ]:
customer_data.fillna(0,inplace=True)

In [ ]:
customer_data.isna().sum()

In [ ]:
zero_trans_customer_data=customer_data.loc[customer_data['freq_transaction']==0]
zero_trans_customer_data

In [ ]:
customer_data_used = customer_data.copy()

# Merge Customer Description

In [ ]:
cust_desc=pd.read_excel('./../data/customer_description_mapping.xlsx').drop(columns=['Unnamed: 0'])
cust_desc

In [ ]:
merged_data = customer_data_used.merge(cust_desc, how='left', right_on='Customer', left_on='Cust No.')
merged_data['Customer'].fillna(0,inplace=True)
merged_data['Customer']=merged_data['Customer'].astype('int64')

merged_data

In [ ]:
merged_data_with=merged_data.drop(columns=['Description'])
merged_data_with_desc=merged_data.copy()

In [ ]:
merged_data_with_desc['Description'].fillna('Unknown', inplace=True)

In [ ]:
merged_data_with_desc.drop(columns=['Customer'],inplace=True)
merged_data_with.drop(columns=['Customer'],inplace=True)

In [ ]:
#tanpa desc
data_clustering=merged_data_with.copy()
data_clustering
# label_encoder = LabelEncoder()
# data_clustering['Description'] = label_encoder.fit_transform(data_clustering['Description'])
# data_clustering

In [ ]:
# data_clustering['Description'].unique()

In [ ]:
scaler = StandardScaler()
columns_to_exclude = ['Cust No.']
columns_to_scale = data_clustering.columns.difference(columns_to_exclude)
customer_data_scaled = data_clustering.copy()
customer_data_scaled[columns_to_scale] = scaler.fit_transform(customer_data_scaled[columns_to_scale])
customer_data_scaled.head()

In [ ]:
customer_data_scaled.set_index('Cust No.', inplace=True)

In [ ]:
customer_data_scaled

In [ ]:
sns.set(style='darkgrid', rc={'axes.facecolor': '#fcf0dc'})

sns.set_palette(['#ff6200'])
km = KMeans(init='k-means++', n_init=10, max_iter=100, random_state=0)
fig, ax = plt.subplots(figsize=(12, 5))
visualizer = KElbowVisualizer(km, k=(2, 15), timings=False, ax=ax)
visualizer.fit(customer_data_scaled)
visualizer.show()

In [ ]:
# Eval
def silhouette_analysis(df, start_k, stop_k, figsize=(15, 16)):
    """
    Perform Silhouette analysis for a range of k values and visualize the results.
    """

    plt.figure(figsize=figsize)
    grid = gridspec.GridSpec(stop_k - start_k + 1, 2)
    first_plot = plt.subplot(grid[0, :])
    sns.set_palette(['darkorange'])

    silhouette_scores = []

    for k in range(start_k, stop_k + 1):
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=100, random_state=0)
        km.fit(df)
        labels = km.predict(df)
        score = silhouette_score(df, labels)
        silhouette_scores.append(score)

    best_k = start_k + silhouette_scores.index(max(silhouette_scores))

    plt.plot(range(start_k, stop_k + 1), silhouette_scores, marker='o')
    plt.xticks(range(start_k, stop_k + 1))
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Silhouette score')
    plt.title('Average Silhouette Score for Different k Values', fontsize=15)

    optimal_k_text = f'The k value with the highest Silhouette score is: {best_k}'
    plt.text(10, 0.23, optimal_k_text, fontsize=12, verticalalignment='bottom', 
             horizontalalignment='left', bbox=dict(facecolor='#fcc36d', edgecolor='#ff6200', boxstyle='round, pad=0.5'))
             

    colors = sns.color_palette("bright")

    for i in range(start_k, stop_k + 1):    
        km = KMeans(n_clusters=i, init='k-means++', n_init=10, max_iter=100, random_state=0)
        row_idx, col_idx = divmod(i - start_k, 2)

        ax = plt.subplot(grid[row_idx + 1, col_idx])

        visualizer = SilhouetteVisualizer(km, colors=colors, ax=ax)
        visualizer.fit(df)

        score = silhouette_score(df, km.labels_)
        ax.text(0.97, 0.02, f'Silhouette Score: {score:.2f}', fontsize=12, \
                ha='right', transform=ax.transAxes, color='red')

        ax.set_title(f'Silhouette Plot for {i} Clusters', fontsize=15)

    plt.tight_layout()
    plt.show()

In [ ]:
silhouette_analysis(customer_data_scaled, 3, 12, figsize=(20, 50))

In [ ]:
kmeans = KMeans(n_clusters=7, init='k-means++', n_init=10, max_iter=300, random_state=1000)
kmeans.fit(customer_data_scaled.drop(columns=['cluster_kmeans']))
cluster_frequencies = Counter(kmeans.labels_)
label_mapping = {label: new_label for new_label, (label, _) in 
                 enumerate(cluster_frequencies.most_common())}
new_labels = np.array([label_mapping[label] for label in kmeans.labels_])
customer_data_scaled['cluster_kmeans'] = new_labels

In [ ]:
customer_data_scaled['cluster_kmeans'].value_counts()

In [ ]:
centroids = customer_data_scaled.groupby('cluster_kmeans').mean()
centroids

In [ ]:
centroids.to_excel('./../results/file/centroid_kmeans_rfm+average_betweetn_trans.xlsx')

In [ ]:
import joblib
joblib.dump(scaler, './../results/scaler_kmeans_v2_without_desc.pkl')

In [ ]:
def find_nearest_cluster(small_cluster, centroids):
    distances = pd.Series(np.linalg.norm(centroids - centroids.loc[small_cluster], axis=1), index=centroids.index)
    nearest_cluster = distances.drop(small_cluster).idxmin()
    return nearest_cluster

In [ ]:
for cluster in [5, 6]:
    nearest_cluster = find_nearest_cluster(cluster, centroids)
    customer_data_scaled['cluster_kmeans'].replace(cluster, nearest_cluster, inplace=True)

In [ ]:
customer_data_scaled['cluster_kmeans'].value_counts()

In [ ]:
# customer_data_scaled['cluster_kmeans'].replace({6: 4}, inplace=True)
# customer_data_scaled['cluster_kmeans'].value_counts()

In [ ]:
data_clustering_result=data_clustering.copy()

In [ ]:
customer_data_scaled.reset_index(inplace=True)

In [ ]:
data_clustering_result

In [ ]:
data_clustering_result['cluster_kmeans']=customer_data_scaled['cluster_kmeans']

In [ ]:
data_pendapatan.drop(columns=['Description'],inplace=True)

In [ ]:
data_with_clusters = data_pendapatan.merge(
    data_clustering_result[['Cust No.', 'cluster_kmeans','Description']],
    on='Cust No.',
    how='left'
)

product_recommendations = data_with_clusters.groupby(['cluster_kmeans', 'Description', 'LEVEL_1'])['Pendapatan in IDR'].sum().reset_index()
product_recommendations = product_recommendations.sort_values(by=['cluster_kmeans', 'Description', 'Pendapatan in IDR'], ascending=[True, True, False])
product_recommendations['Description'] = product_recommendations['Description'].astype(int)
product_recommendations['Description']=label_encoder.inverse_transform(product_recommendations['Description'])
product_recommendations.to_excel('./../results/file/product_statistics_level_1_each_cluster.xlsx')


In [ ]:
data_clustering_result_dump=data_clustering_result.copy()
data_clustering_result_dump

In [ ]:
data_clustering_result_dump['cluster_kmeans'].value_counts()

In [ ]:
data_clustering_result_dump['Description']=label_encoder.inverse_transform(data_clustering_result_dump['Description'])
data_clustering_result_dump

In [ ]:
data_clustering_result_dump.to_excel('./../results/customer_df_with_clusters_kmeans_without_desc.xlsx')

In [ ]:
data_clustering_result_dump['cluster_kmeans'].value_counts()

In [ ]:
testing_dump=pd.read_excel('./../data/mapping_customer_name_customer_no_pendapatan.xlsx')
testing_dump

In [ ]:
data_with_clusters = data_pendapatan.merge(
    data_clustering_result[['Cust No.', 'cluster_kmeans','Description']],
    on='Cust No.',
    how='left'
)

product_recommendations = data_with_clusters.groupby(['cluster_kmeans', 'Description', 'LEVEL_1'])['Pendapatan in IDR'].sum().reset_index()
product_recommendations = product_recommendations.sort_values(by=['cluster_kmeans', 'Description', 'Pendapatan in IDR'], ascending=[True, True, False])
product_recommendations['Description'] = product_recommendations['Description'].astype(int)
product_recommendations['Description']=label_encoder.inverse_transform(product_recommendations['Description'])
# product_recommendations.to_excel('./../results/file/product_statistics_level_2_each_cluster.xlsx')
product_recommendations

In [ ]:
data_with_clusters = data_pendapatan.merge(
    data_clustering_result[['Cust No.', 'cluster_kmeans','Description']],
    on='Cust No.',
    how='left'
)

product_recommendations = data_with_clusters.groupby(['cluster_kmeans', 'Description', 'LEVEL_2'])['Pendapatan in IDR'].sum().reset_index()
product_recommendations = product_recommendations.sort_values(by=['cluster_kmeans', 'Description', 'Pendapatan in IDR'], ascending=[True, True, False])
product_recommendations['Description'] = product_recommendations['Description'].astype(int)
product_recommendations['Description']=label_encoder.inverse_transform(product_recommendations['Description'])
product_recommendations.to_excel('./../results/file/product_statistics_level_2_each_cluster.xlsx')
product_recommendations

In [ ]:
data_with_clusters = data_pendapatan.merge(
    data_clustering_result[['Cust No.', 'cluster_kmeans','Description']],
    on='Cust No.',
    how='left'
)

product_recommendations = data_with_clusters.groupby(['cluster_kmeans', 'Description', 'LEVEL_3'])['Pendapatan in IDR'].sum().reset_index()
product_recommendations = product_recommendations.sort_values(by=['cluster_kmeans', 'Description', 'Pendapatan in IDR'], ascending=[True, True, False])
product_recommendations['Description'] = product_recommendations['Description'].astype(int)
product_recommendations['Description']=label_encoder.inverse_transform(product_recommendations['Description'])
product_recommendations.to_excel('./../results/file/product_statistics_level_3_each_cluster.xlsx')
product_recommendations

# Statistical Cluster

In [ ]:
data_with_clusters = data_pendapatan_used.merge(
    data_clustering_result[['Cust No.', 'cluster_kmeans']],
    on='Cust No.',
    how='left'
)
data_with_clusters
# data_with_clusters.rename(columns={'Description_x':'Description','Description_y':'Customer Type'},inplace=True)

# data_with_clusters['Customer Type']=label_encoder.inverse_transform(data_with_clusters['Customer Type'])
# data_with_clusters

In [ ]:
sns.set_style('whitegrid')
colors = ['#ff6200', '#ffcaa8', '#ffcaa5', '#ffcaa8', '#ff6200']
my_cmap = LinearSegmentedColormap.from_list('custom_map', colors, N=256)

In [ ]:
data_clustering_result_viz=data_clustering_result.copy()
data_clustering_result_viz
# data_clustering_result_viz['Description']=label_encoder.inverse_transform(data_clustering_result_viz['Description'])
# data_clustering_result_viz

In [ ]:
data_clustering_result_viz

In [ ]:
a=merged_data_with_desc[['Cust No.','Description']].copy()
b=data_clustering_result_viz.merge(a,how='left',on='Cust No.')

In [ ]:
b

In [ ]:
cluster_percentage = (b['Description'].value_counts(normalize=True) * 100).reset_index()
cluster_percentage.columns = ['Description', 'Percentage']
cluster_percentage.sort_values(by='Description', inplace=True)
plt.figure(figsize=(10, 4))
sns.barplot(x='Percentage', y='Description', data=cluster_percentage, orient='h', palette=colors)

for index, value in enumerate(cluster_percentage['Percentage']):
    plt.text(value+0.5, index, f'{value:.2f}%')

plt.title('Distribution of Description', fontsize=14)
plt.xticks(ticks=np.arange(0, 50, 5))
plt.xlabel('Percentage (%)')
plt.show()

In [ ]:
cluster_description_count = b.groupby(['cluster_kmeans', 'Description']).size().reset_index(name='Count')
cluster_total = cluster_description_count.groupby('cluster_kmeans')['Count'].transform('sum')
cluster_description_count['Percentage'] = (cluster_description_count['Count'] / cluster_total) * 100
plt.figure(figsize=(14, 7))
sns.barplot(
    x='cluster_kmeans',
    y='Percentage',
    hue='Description',
    data=cluster_description_count,
    palette='Set2'
)
for index, row in cluster_description_count.iterrows():
    plt.text(
        row['cluster_kmeans'],
        row['Percentage'] + 0.5,
        f"{row['Percentage']:.1f}%",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.title('Distribution of Customers and Description Across Clusters', fontsize=14)
plt.xlabel('Cluster')
plt.ylabel('Percentage (%)')
plt.xticks(fontsize=12)
plt.legend(title='Description', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
cluster_percentage = (data_clustering_result_viz['cluster_kmeans'].value_counts(normalize=True) * 100).reset_index()
cluster_percentage.columns = ['Cluster', 'Percentage']
cluster_percentage.sort_values(by='Cluster', inplace=True)
plt.figure(figsize=(10, 4))
sns.barplot(x='Percentage', y='Cluster', data=cluster_percentage, orient='h', palette=colors)

for index, value in enumerate(cluster_percentage['Percentage']):
    plt.text(value+0.5, index, f'{value:.2f}%')

plt.title('Distribution of Customers Across Clusters', fontsize=14)
plt.xticks(ticks=np.arange(0, 50, 5))
plt.xlabel('Percentage (%)')
plt.show()

In [ ]:
data_clustering_result_radar=data_clustering_result.copy()

In [ ]:
data_clustering_result_radar

In [ ]:
data_clustering_result_radar = data_clustering_result_radar.set_index('Cust No.')

In [ ]:
scaler_new = StandardScaler()
df_customer_standardized = scaler.fit_transform(data_clustering_result_radar.drop(columns=['cluster_kmeans'], axis=1))
df_customer_standardized = pd.DataFrame(df_customer_standardized, columns=data_clustering_result_radar.columns[:-1], index=data_clustering_result_radar.index)
df_customer_standardized['cluster'] = data_clustering_result_radar['cluster_kmeans']
# df_customer_standardized=df_customer_standardized.drop(columns=['Description'])
cluster_centroids = df_customer_standardized.groupby('cluster').mean()

def create_radar_chart(ax, angles, data, color, cluster):
    ax.fill(angles, data, color=color, alpha=0.4)
    ax.plot(angles, data, color=color, linewidth=2, linestyle='solid')
    ax.set_title(f'Cluster {cluster}', size=16, color=color, y=1.1)

labels = np.array(cluster_centroids.columns)
num_vars = len(labels)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
labels = np.concatenate((labels, [labels[0]]))
angles += angles[:1]

colors = ['b', 'g', 'r', 'c', 'm']

rows, cols = 5, 1
fig, ax = plt.subplots(figsize=(16, 20), subplot_kw=dict(polar=True), nrows=rows, ncols=cols)

ax = ax.flatten()

for i, color in enumerate(colors[:len(cluster_centroids)]):
    data = cluster_centroids.loc[i].tolist()
    data += data[:1]
    create_radar_chart(ax[i], angles, data, color, i)

    ax[i].set_xticks(angles[:-1])
    ax[i].set_xticklabels(labels[:-1], fontsize=10)

    ax[i].grid(color='grey', linewidth=0.5)
    ax[i].set_yticklabels([])
plt.tight_layout()
plt.savefig('./../results/plot/cluster_results_profile_v4.png',dpi=300, bbox_inches='tight')
plt.show()


# Association Rules

In [ ]:
data_with_clusters = data_pendapatan_used.merge(
    data_clustering_result[['Cust No.', 'cluster_kmeans']],
    on='Cust No.',
    how='left'
)

In [ ]:
data_with_clusters.columns

In [ ]:
data_with_clusters.rename(columns={'Description_x':'Description','Description_y':'Customer Type'},inplace=True)

In [ ]:
data_with_clusters['Customer Type']=label_encoder.inverse_transform(data_with_clusters['Customer Type'])
data_with_clusters

In [ ]:
data_with_clusters.to_csv('./../data/row_data_pendapatan_with_cluster.csv')

In [ ]:
unique_level_1 = data_with_clusters['LEVEL_1'].dropna().unique().tolist()
unique_level_2 = data_with_clusters['LEVEL_2'].dropna().unique().tolist()
unique_level_3 = data_with_clusters['LEVEL_3'].dropna().unique().tolist()

unique_levels = {
    'LEVEL_1': unique_level_1,
    'LEVEL_2': unique_level_2,
    'LEVEL_3': unique_level_3
}

In [ ]:
import json
with open('./../data/list_unique_product_levels.json', 'w', encoding='utf-8') as f:
    json.dump(unique_levels, f, indent=4, ensure_ascii=False)


In [ ]:
# rules level 3
from mlxtend.frequent_patterns import apriori, association_rules
cluster_data = data_with_clusters[data_with_clusters['cluster_kmeans'] == 4]
basket = cluster_data.pivot_table(index='Cust No.', columns='LEVEL_3', aggfunc='size', fill_value=0)
basket = basket.applymap(lambda x: 1 if x > 0 else 0)
frequent_itemsets = apriori(basket, min_support=0.1, use_colnames=True,low_memory=True)
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
rules
# rules.to_excel(f'./../results/ass_rules/level_3/cluster_{4}.xlsx')

In [ ]:
data_with_clusters.columns

In [ ]:
# rules level 2
from mlxtend.frequent_patterns import apriori, association_rules
# cluster_data = data_with_clusters[data_with_clusters['cluster_kmeans'] == i]
# basket = cluster_data.groupby(['Cust No.', 'Invoice No.', 'LEVEL_2']).size().reset_index(name='Count')
# basket = basket.pivot_table(index='Invoice No.', columns='LEVEL_2', aggfunc='size', fill_value=0)
# basket = basket.applymap(lambda x: 1 if x > 0 else 0)

for i in range(4):
    cluster_data = data_with_clusters[data_with_clusters['cluster_kmeans'] == i]
    basket = cluster_data.groupby(['Cust No.', 'Nomor Nota', 'LEVEL_2']).size().reset_index(name='Count')
    basket = basket.pivot_table(index='Nomor Nota', columns='LEVEL_2', aggfunc='size', fill_value=0)
    basket = basket.applymap(lambda x: 1 if x > 0 else 0)
    # cluster_data = data_with_clusters[data_with_clusters['cluster_kmeans'] == i]
    # basket = cluster_data.pivot_table(index='Cust No.', columns='LEVEL_2', aggfunc='size', fill_value=0)
    # basket = basket.applymap(lambda x: 1 if x > 0 else 0)
    frequent_itemsets = apriori(basket, min_support=0.1, use_colnames=True)
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    rules.to_excel(f'./../results/ass_rules/level_2/cluster_v1_{i}.xlsx')

In [ ]:
customer_purchases_unique = data_with_clusters.groupby('Cust No.')['LEVEL_2'].unique().apply(list).to_dict()
customer_purchases_unique

In [ ]:
import os
import json
output_path = './../results/file/customer_purchases_unique_level_2.json'
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(customer_purchases_unique, f, indent=4, ensure_ascii=False)

# Trash

In [ ]:
# rules level 1
from mlxtend.frequent_patterns import apriori, association_rules
for i in range(5):
    cluster_data = data_with_clusters[data_with_clusters['cluster_kmeans'] == i]
    basket = cluster_data.pivot_table(index='Cust No.', columns='LEVEL_1', aggfunc='size', fill_value=0)
    basket = basket.applymap(lambda x: 1 if x > 0 else 0)
    frequent_itemsets = apriori(basket, min_support=0.1, use_colnames=True)
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)
    rules.to_excel(f'./../results/ass_rules/level_1/cluster_{i}.xlsx')

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

cluster_rules = {}

for cluster in data_with_clusters['cluster_kmeans'].unique():
    print(f"\nGenerating Association Rules for Cluster {cluster}:")
    cluster_data = data_with_clusters[data_with_clusters['cluster_kmeans'] == cluster]
    basket = cluster_data.pivot_table(index='Cust No.', columns='LEVEL_3', aggfunc='size', fill_value=0)
    basket = basket.applymap(lambda x: 1 if x > 0 else 0)
    frequent_itemsets = apriori(basket, min_support=0.1, use_colnames=True)
    rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.0)    
    cluster_rules[cluster] = rules

In [ ]:
def recommend_product_hierarchy(purchased_products, cluster, cluster_rules, level='Level_2'):
    if cluster not in cluster_rules:
        return "No Rules For this cluster."
    rules = cluster_rules[cluster]
    recommendations = []
    for product in purchased_products:
        matching_rules = rules[rules['antecedents'] == frozenset([product])]
        for _, rule in matching_rules.iterrows():
            recommendations.extend(list(rule['consequents']))
    recommendations = list(set(recommendations))
    if recommendations:
        return recommendations
    else:
        return "No Recommendations based on the product bought."